# lfp_lzc_analyser

In [51]:
# Használt könyvtárak
import sys
import os
import numpy as np
import h5py as hp
import pandas as pd
import time
import re
from neurokit2.complexity import complexity_lempelziv

In [71]:
# LZC analízis + eredmények mentése új mappába (.npy, .xlsx)
    
def lfp_lzc_analyser(input_filename, channel_indices_file=None, max_channels=None):
    """
    Fő függvény ami futtatja az LZC analízist
    input_filename: az elemzendő fájl elérési útvonala
    channel_indices_file: opcionális, az eredeti csatorna indexek fájlja
    max_channels: opcionális, hány csatornára fusson (None = összes)
    """

    # =====================
    # 1. KÖNYVTÁR LÉTREHOZÁSA EREDMÉNYEKNEK
    # =====================

    # Eredmény mappa neve a fájlnév alapján - ABSZOLÚT ÚTVONAL
    base_name = os.path.splitext(os.path.basename(input_filename))[0]
    input_dir = os.path.dirname(input_filename)
    
    # Mappa nevéből minden nem kívánt karaktert cseréljünk
    safe_base_name = "".join(c if c.isalnum() or c in (' ', '_', '-') else '_' for c in base_name)
    safe_base_name = safe_base_name.replace(' ', '_')
    
    # ABSZOLÚT útvonal használata
    results_dir = os.path.join(input_dir, f"{safe_base_name}_LZC_results")

    # Ha létezik, megállítjuk a scriptet
    if os.path.exists(results_dir):
        print(f"Error: results directory: {results_dir}, already exists.")
        sys.exit()

    # új, üres mappa létrehozása
    os.makedirs(results_dir)
    print(f"New results directory is made: {results_dir}")

    # Csatorna indexek betöltése, ha meg van adva
    channel_indices = None
    if channel_indices_file and os.path.exists(channel_indices_file):
        channel_indices = np.load(channel_indices_file)
        print(f" Channel indices loaded: {len(channel_indices)} indices")
    elif channel_indices_file:
        print(f" Warning: Channel indices file not found: {channel_indices_file}")


   # =====================
   # 2. ADAT BETÖLTÉSE
   # =====================

    print(f" Loading data: {input_filename}")
    LFP_data = np.load(input_filename)
    print(f" Data loaded: {LFP_data.shape}")
    
    # Max channels beállítása
    if max_channels is None:
        max_channels = LFP_data.shape[1]
        print(f" Processing all {max_channels} channels")
    else:
        max_channels = min(max_channels, LFP_data.shape[1])
        print(f" Processing {max_channels} out of {LFP_data.shape[1]} channels")


   # =====================
   # 3. LZC SZÁMÍTÁS MINDEN CSATORNÁRA
   # =====================

    def calculate_lzc_all_channels(data, max_channels):  
        num_channels = min(data.shape[1], max_channels)
        lzc_results = np.zeros(num_channels)
        lzc_details = []  # A dictionary-k tárolására
        
        print(f"Calculating LZC on {num_channels} channels...")
        start_time = time.time()
        print(f"Analysis started at: {time.ctime(start_time)}")
        
        for i in range(num_channels):
            print(f"Channel {i+1}/{num_channels} is beeing processed...")
            channel_data = data[:, i]
            result = complexity_lempelziv(channel_data)
            
            # A függvény (LZC_érték, dictionary) tuple-t ad vissza
            lzc_value = result[0]  					# Csak az LZC számérték
            details_dict = result[1]  				# A dictionary rész
            lzc_results[i] = lzc_value
            lzc_details.append(details_dict)  		# Dictionary hozzáadása
            
            # Progressz indikátor
            elapsed = time.time() - start_time
            print(f"Channel {i+1} done: LZC = {lzc_value:.6f} ({elapsed:.1f}s)")
        
        total_time = time.time() - start_time
        print(f"DONE! Total time: {total_time:.1f} seconds")
        print(f"Average LZC: {np.mean(lzc_results):.6f}")
        
        return lzc_results, lzc_details, start_time  # 3 értéket ad vissza

    # Futatás a megadott számú csatornára
    lzc_values, lzc_dicts, analysis_start_time = calculate_lzc_all_channels(LFP_data, max_channels=max_channels)

    # Eredmények részletesen
    print(f"\n Results:")
    for i, (lzc_val, details) in enumerate(zip(lzc_values, lzc_dicts)):
        channel_idx = channel_indices[i] if channel_indices is not None else i
        print(f"   Channel {channel_idx}: LZC = {lzc_val:.6f}")
        print(f"      Dictionary: {details}")	


   # =====================
   # 4. ÁTFOGÓ LZC ELEMZÉS
   # =====================

    def comprehensive_lzc_analysis(lzc_values):
        
        print("\n LZC Analysis - Summary")
        print("=" * 100)
        print(f"Average LZC: {np.mean(lzc_values):.6f}")
        print(f"Median LZC: {np.median(lzc_values):.6f}")
        print(f"LZC standard deviation: {np.std(lzc_values):.6f}")
        print(f"LZC range: {np.min(lzc_values):.6f} - {np.max(lzc_values):.6f}")
        print(f"Analysed channels: {len(lzc_values)}")
        return {
            'mean': np.mean(lzc_values),
            'median': np.median(lzc_values),
            'std': np.std(lzc_values),
            'min': np.min(lzc_values),
            'max': np.max(lzc_values),
            'channels_analyzed': len(lzc_values)
        }

    # Elemzés futtatása
    summary = comprehensive_lzc_analysis(lzc_values)


   # =====================
   # 5. EREDMÉNYEK MENTÉSE (numpy, excel) - ÚJ MAPPÁBA
   # =====================

    print(f"\n Saving results to '{results_dir}' directory...")

    # --- LZC értékek mentése numpy formátumban ---
    lzc_npy_path = os.path.join(results_dir, f'{base_name}_LZC_values.npy')
    np.save(lzc_npy_path, lzc_values)
    print(f" LZC values saved: {lzc_npy_path}")

    # --- DataFrame-ek létrehozása ---
    if channel_indices is not None:
        # Ha vannak csatorna indexek, azokat használjuk + dictionary adatok
        lzc_df = pd.DataFrame({
            'original_channel_index': channel_indices[:len(lzc_values)],
            'lzc_value': lzc_values,
            'permutation': [d['Permutation'] for d in lzc_dicts],
            'complexity_kolmogorov': [d['Complexity_Kolmogorov'] for d in lzc_dicts]
        })
    else:
        # Egyébként sima indexelés + dictionary adatok
        lzc_df = pd.DataFrame({
            'channel_index': range(len(lzc_values)),
            'lzc_value': lzc_values,
            'permutation': [d['Permutation'] for d in lzc_dicts],
            'complexity_kolmogorov': [d['Complexity_Kolmogorov'] for d in lzc_dicts]
        })

    # Summary bővítése számítási idővel
    total_computation_time = time.time() - analysis_start_time
    summary['total_computation_time_seconds'] = total_computation_time
    summary['total_computation_time_hours'] = total_computation_time / 3600
    
    summary_df = pd.DataFrame([summary])

    # --- Excel fájl mentése ---
    excel_filename = os.path.join(results_dir, f'{base_name}_LZC_results.xlsx')

    with pd.ExcelWriter(excel_filename, engine='openpyxl') as writer:
        lzc_df.to_excel(writer, sheet_name='LZC_values', index=False)
        summary_df.to_excel(writer, sheet_name='Summary', index=False)

    print(f" Excel file saved: {excel_filename}")

    print(f"\n ALL DONE!")
    return lzc_values, summary


# =====================
# 6. Az analízis függvény indításkezelő része (konzol + jupyter notebook)
# =====================

if __name__ == "__main__":
    # Fájlnév megadása konzolból
    if len(sys.argv) > 1:
        input_filename = sys.argv[1]
        # Opcionális csatorna index fájl (második argumentum)
        channel_indices_file = sys.argv[2] if len(sys.argv) > 2 else None
        # Opcionális max channels (harmadik argumentum)
        max_channels = int(sys.argv[3]) if len(sys.argv) > 3 else None
        lfp_lzc_analyser(input_filename, channel_indices_file, max_channels)
    else:
        # Notebook használatra is
        print("No command line arguments. You can call the function manually:")
        print("Example: lfp_lzc_analyser('your_file.npy', max_channels=10)")
        print("Or run from terminal: python script.py your_file.npy")


OSError: [WinError 433] Nem létező eszközt adtak meg: '-f_LZC_results'

In [79]:
# =====================
# 6. Futtatás jupyter notebook-ból (Ebbe kell belerakni az aktuális adatokat!!!)
# =====================

lzc_results, summary = lfp_lzc_analyser(
    r'D:\LZC_Analysis\Pt02_LZC\Pt02_LFP_20nth_3rd_third.npy',
    r'D:/LZC_Analysis/Pt02_LZC/Pt02_selected_channel_indices.npy'
)

New results directory is made: D:\LZC_Analysis\Pt02_LZC\Pt02_LFP_20nth_3rd_third_LZC_results
 Channel indices loaded: 20 indices
 Loading data: D:\LZC_Analysis\Pt02_LZC\Pt02_LFP_20nth_3rd_third.npy
 Data loaded: (268750, 20)
 Processing all 20 channels
Calculating LZC on 20 channels...
Analysis started at: Thu Nov 27 00:42:45 2025
Channel 1/20 is beeing processed...
Channel 1 done: LZC = 0.098652 (1969.8s)
Channel 2/20 is beeing processed...
Channel 2 done: LZC = 0.076573 (3850.4s)
Channel 3/20 is beeing processed...
Channel 3 done: LZC = 0.102880 (5377.2s)
Channel 4/20 is beeing processed...
Channel 4 done: LZC = 0.087579 (7082.1s)
Channel 5/20 is beeing processed...
Channel 5 done: LZC = 0.112208 (8601.4s)
Channel 6/20 is beeing processed...
Channel 6 done: LZC = 0.085968 (10371.6s)
Channel 7/20 is beeing processed...
Channel 7 done: LZC = 0.085901 (12240.8s)
Channel 8/20 is beeing processed...
Channel 8 done: LZC = 0.109390 (13844.5s)
Channel 9/20 is beeing processed...
Channel 9 do

## Konzolos futtatás
1. Windows:
    - Win + R, majd írd be: cmd
    - Vagy keresd a "Command Prompt"-ot (anaconda prompt)
   
3. Futtasd a scriptet: (bárhonnan mert abs. utak)
    - python "D:\Python Scripts\lzc_analyser.py" "D:\LZC_Analysis\Pt02_LZC\Pt02_LFP_20nth_1st_third.npy" "D:\LZC_Analysis\Pt02_LZC\Pt02_selected_channel_indices.npy"